In [ ]:
import requests
import time
import base64
from PIL import Image

In [ ]:
#sanity check with /v1/models

# enter the inference endpoints/url from the deployed model
#example of base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"
#base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"
base_url = "<enter your base url here>"
requests.get(f"{base_url}/v1/models").json()

In [ ]:
#Perform a basic text-only test.  Remember that you are using the cpu to process the request and it can take up to 30s

resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [{"role": "user", "content": "Hello, are you working?"}],
        "max_tokens": 100
    }
)
print(resp.status_code)
print(resp.json())

In [ ]:
#we have copied a chart from the keycloaks docs pdf for testing.  Let's do a sanity check on it.
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_b64 = encode_image("chart.png")
print("Encoded length:", len(image_b64))  # sanity check — should be a long string, not empty

In [ ]:
#Perform a multimodal test - send the chart image + question.  Be patient as we are using the cpu and this can take up to 2 mins to execute.

#Note:  if you see a long response time of 162s or greater, rerun this cell to see if you can get a faster response e.g. 40s

#If you see such a huge drop (in your second run) it is based on a cold-start overhead (torch-compile warmup, cache population, CPU thread affinity settling) 
#on a freshly-restarted pod. After the first run, the pod is now "warm," and inference speed will be normalized to something even better than the original ~162s baseline.

# Check image size first
img = Image.open("chart.png")
print("Image size:", img.size)

#For a 619x344 image on CPU-only inference the range should 30-90 seconds for inference response.
start = time.time()
resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": "What does this chart show?"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}
        ],
        "max_tokens": 100
    },
    timeout=300
)
print("Elapsed:", time.time() - start, "seconds")
print(resp.status_code)
print(resp.json())

In [ ]:
#let's print out the content so that we can easily view it

print(resp.json()['choices'][0]['message']['content'])


<b>Let's talk Response Time </b>
The response quality (60-90s) is genuinely good — it correctly identified the image as an architecture diagram, described the high-availability/replication concept, and mentioned load balancers and data center replication — that's real diagram comprehension, not a generic answer.

In [ ]:
#test the embedding model

# Perform a basic embedding test
# enter the inference endpoints/url from the deployed model
#example of embed_base_url = "http://redhataiall-minilm-l6-v2-predictor.user9.svc.cluster.local"

embed_base_url = "<enter your base url here>"

resp = requests.post(
    f"{embed_base_url}/v1/embeddings",
    json={
        "model": "redhataiall-minilm-l6-v2",
        "input": "Hello, are you working?"
    }
)

print(resp.status_code)

data = resp.json()
embedding = data["data"][0]["embedding"]
print("Embedding length:", len(embedding))       # should print 384
print("First 5 values:", embedding[:5])

In [ ]:
import numpy as np

def get_embedding(text):
    resp = requests.post(
        f"{embed_base_url}/v1/embeddings",
        json={"model": "redhataiall-minilm-l6-v2", "input": text}
    )
    return np.array(resp.json()["data"][0]["embedding"])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

e1 = get_embedding("The cat sat on the mat.")
e2 = get_embedding("A feline rested on the rug.")     # similar meaning
e3 = get_embedding("The stock market crashed today.")  # unrelated meaning

print("Similar sentences:", cosine_similarity(e1, e2))    # should be high, e.g. 0.6-0.9
print("Unrelated sentences:", cosine_similarity(e1, e3))  # should be much lower

<b>Test semantic similarity to prove the embedding  model is working properly.</b>

Confirms the model actually understands meaning, not just that the API responds.

In our example, the "similar sentences" score comes back noticeably higher than the "unrelated sentences" score, that's solid confirmation the embedding model is working correctly end-to-end.